Fraud Detection - Shipment/Return Round-Trip Analysis
Identifies month-end shipments reversed as returns shortly after, using exact lot matching. Flags candidates for human review, not automated accusation.

In [0]:
%run ../../_local_config

In [0]:
import sys
sys.path.append("/Workspace/Users/venura-it@brownsgroup.com/Exide sales/Exide-Sales-Forecast")

from src.io.storage import get_blob_service, read_silver, save_gold
from src.analysis.fraud_detection import match_shipment_return_pairs
import pandas as pd
import io

blob_service = get_blob_service(storage_account_name, storage_account_key)

ANALYSIS_BASE = "live/battery/analysis"

Load analysis silver

In [0]:
analysis_silver = read_silver(blob_service, f"{ANALYSIS_BASE}/battery_analysis_clean_live.json")
analysis_silver["posting_date"] = pd.to_datetime(analysis_silver["posting_date"])
print(f"Analysis silver: {analysis_silver.shape}")

Find exact-lot matches

In [0]:
lot_matches = match_shipment_return_pairs(analysis_silver)
print(f"Exact-lot matched shipment/return pairs: {len(lot_matches)}")
lot_matches["days_between"] = (lot_matches["posting_date_return"] - lot_matches["posting_date_shipment"]).dt.days
print(lot_matches[["item_no", "lot_no", "sales_person_code", "posting_date_shipment", "posting_date_return", "days_between"]].head(20))